In [3]:
import cv2
import numpy as np
import os
from ultralytics import YOLO

# 1. CARGAR AMBOS MODELOS (El poder del procesamiento en paralelo)
print("Cargando Cerebro 1 (Segmentación COCO)...")
model_seg = YOLO('yolov8n-seg.pt') 

print("Cargando Cerebro 2 (Tu Modelo Personalizado para Señales)...")
model_custom = YOLO('../models/entrenamiento_script_estable/weights/best.pt')

os.makedirs('../reports/alertas_visuales', exist_ok=True)

ruta_video_entrada = '../test/MedioDiaLluvia.mov' 
ruta_video_salida = '../reports/alertas_visuales/MedioDiaLluvia_segmentacion.mp4'

cap = cv2.VideoCapture(ruta_video_entrada)

if not cap.isOpened():
    print("Error abriendo el video.")
else:
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    out = cv2.VideoWriter(ruta_video_salida, cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

    colores_seg = {
        0: (0, 0, 255),    # Persona -> Rojo
        1: (0, 165, 255),  # Bici -> Naranja
        2: (255, 255, 0),  # Coche -> Cian
        3: (0, 0, 255),    # Moto -> Rojo
        5: (255, 255, 0),  # Autobús -> Cian
        7: (255, 255, 0)   # Camión -> Cian
    }

    print("Iniciando procesamiento Multi-Modelo...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # --- FASE 1: SILUETAS CON MODELO DE SEGMENTACIÓN ---
        res_seg = model_seg.predict(frame, classes=[0, 1, 2, 3, 5, 7], conf=0.4, verbose=False)
        capa_overlay = frame.copy()
        
        if res_seg[0].masks is not None:
            mascaras = res_seg[0].masks.xy
            clases_seg = res_seg[0].boxes.cls.cpu().numpy()
            
            for mascara, clase_id in zip(mascaras, clases_seg):
                color_bgr = colores_seg.get(int(clase_id), (0, 255, 0)) 
                puntos_poligono = np.array(mascara, np.int32)
                cv2.fillPoly(capa_overlay, [puntos_poligono], color_bgr)
                cv2.polylines(frame, [puntos_poligono], True, color=color_bgr, thickness=2)

        # Aplicar transparencia de las máscaras
        frame_final = cv2.addWeighted(capa_overlay, 0.45, frame, 0.55, 0)

        # --- FASE 2: CAJAS DE SEÑALES CON TU MODELO PERSONALIZADO ---
        # Le decimos que SOLO busque la clase 3 (Señales) de tu dataset
        res_custom = model_custom.predict(frame, classes=[3], conf=0.4, verbose=False)
        
        for caja in res_custom[0].boxes:
            x1, y1, x2, y2 = map(int, caja.xyxy[0].tolist())
            confianza = float(caja.conf[0])
            
            # Dibujamos un cuadro amarillo para las señales (sin transparencia, para que resalte)
            cv2.rectangle(frame_final, (x1, y1), (x2, y2), (0, 255, 255), 2)
            
            texto_senal = f"Senal {confianza*100:.0f}%"
            (w, h), _ = cv2.getTextSize(texto_senal, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame_final, (x1, y1 - 20), (x1 + w, y1), (0, 255, 255), -1)
            cv2.putText(frame_final, texto_senal, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

        # Guardar el frame con ambos análisis
        out.write(frame_final)

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"¡Video Híbrido Finalizado! Revisa: {ruta_video_salida}")

Cargando Cerebro 1 (Segmentación COCO)...
Cargando Cerebro 2 (Tu Modelo Personalizado para Señales)...
Iniciando procesamiento Multi-Modelo...
¡Video Híbrido Finalizado! Revisa: ../reports/alertas_visuales/MedioDiaLluvia_segmentacion.mp4


In [1]:
import cv2
import numpy as np
import os
import time  # NUEVO: Importamos time para medir FPS y Latencia
from ultralytics import YOLO

# 1. CARGAR AMBOS MODELOS
print("Cargando Cerebro 1 (Segmentación COCO)...")
model_seg = YOLO('yolov8n-seg.pt') 

print("Cargando Cerebro 2 (Tu Modelo Personalizado para Señales)...")
model_custom = YOLO('../models/entrenamiento_script_estable/weights/best.pt')

os.makedirs('../reports/alertas_visuales', exist_ok=True)

ruta_video_entrada = '../test/dia2.mov' 
ruta_video_salida = '../reports/alertas_visuales/dia2segmentacion.mp4'

cap = cv2.VideoCapture(ruta_video_entrada)

if not cap.isOpened():
    print("Error abriendo el video.")
else:
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_video = int(cap.get(cv2.CAP_PROP_FPS))
    
    # Area total de la pantalla para calcular porcentajes
    area_pantalla = ancho * alto
    
    out = cv2.VideoWriter(ruta_video_salida, cv2.VideoWriter_fourcc(*'mp4v'), fps_video, (ancho, alto))

    # Colores base (se sobrescribirán por la heurística de riesgo)
    colores_seg = {
        0: (255, 0, 0),    # Persona -> Azul oscuro (BGR)
        1: (255, 165, 0),  # Bici -> Azul claro
        2: (255, 255, 0),  # Coche -> Cian
        3: (255, 0, 0),    # Moto -> Azul oscuro
        5: (255, 255, 0),  # Autobús -> Cian
        7: (255, 255, 0)   # Camión -> Cian
    }

    print("Iniciando procesamiento Multi-Modelo con Heurística de Riesgo...")

    while cap.isOpened():
        # NUEVO: Empezamos a medir el tiempo del frame
        start_time = time.time()
        
        ret, frame = cap.read()
        if not ret: break

        capa_overlay = frame.copy()
        
        # Variables para el estado de riesgo global del frame
        riesgo_global_frame = "BAJO"
        color_alerta_global = (0, 255, 0) # Verde

        # --- FASE 1: SILUETAS CON MODELO DE SEGMENTACIÓN ---
        res_seg = model_seg.predict(frame, classes=[0, 1, 2, 3, 5, 7], conf=0.4, verbose=False)
        
        if res_seg[0].masks is not None:
            mascaras = res_seg[0].masks.xy
            cajas_seg = res_seg[0].boxes # Extraemos las cajas para calcular el área
            clases_seg = res_seg[0].boxes.cls.cpu().numpy()
            
            for mascara, caja, clase_id in zip(mascaras, cajas_seg, clases_seg):
                clase_int = int(clase_id)
                color_bgr = colores_seg.get(clase_int, (0, 255, 0))
                
                # --- LÓGICA ESPACIAL (ÁREA Y RIESGO) ---
                ancho_caja = caja.xywh[0][2].item()
                alto_caja = caja.xywh[0][3].item()
                area_caja = ancho_caja * alto_caja
                porcentaje_area = (area_caja / area_pantalla) * 100
                
                # Reglas de Heurística (Las de la tabla de tu reporte)
                es_vulnerable = clase_int in [0, 1, 3] # Peatón, Bici, Moto
                
                if es_vulnerable and porcentaje_area > 5.0:
                    color_bgr = (0, 0, 255) # ROJO (Crítico)
                    riesgo_global_frame = "CRÍTICO"
                elif not es_vulnerable and porcentaje_area > 15.0:
                    color_bgr = (0, 0, 255) # ROJO (Crítico)
                    riesgo_global_frame = "CRÍTICO"
                elif porcentaje_area > 8.0:
                    color_bgr = (0, 165, 255) # NARANJA (Medio)
                    if riesgo_global_frame != "CRÍTICO":
                        riesgo_global_frame = "MEDIO"

                # Dibujar máscara con el color del riesgo
                puntos_poligono = np.array(mascara, np.int32)
                cv2.fillPoly(capa_overlay, [puntos_poligono], color_bgr)
                cv2.polylines(frame, [puntos_poligono], True, color=color_bgr, thickness=2)

        # Aplicar transparencia de las máscaras
        frame_final = cv2.addWeighted(capa_overlay, 0.45, frame, 0.55, 0)

        # --- FASE 2: CAJAS DE SEÑALES CON TU MODELO PERSONALIZADO ---
        res_custom = model_custom.predict(frame, classes=[3], conf=0.4, verbose=False)
        
        for caja in res_custom[0].boxes:
            x1, y1, x2, y2 = map(int, caja.xyxy[0].tolist())
            confianza = float(caja.conf[0])
            
            cv2.rectangle(frame_final, (x1, y1), (x2, y2), (0, 255, 255), 2)
            texto_senal = f"Senal {confianza*100:.0f}%"
            (w, h), _ = cv2.getTextSize(texto_senal, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame_final, (x1, y1 - 20), (x1 + w, y1), (0, 255, 255), -1)
            cv2.putText(frame_final, texto_senal, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

        # --- FASE 3: CALCULAR RENDIMIENTO Y DIBUJAR HUD (TELEMETRÍA) ---
        end_time = time.time()
        latencia = (end_time - start_time) * 1000 # En milisegundos
        fps_actuales = 1.0 / (end_time - start_time)
        
        # Color del texto de alerta global
        if riesgo_global_frame == "CRÍTICO": color_alerta_global = (0, 0, 255)
        elif riesgo_global_frame == "MEDIO": color_alerta_global = (0, 165, 255)
        
        # Dibujar Panel Superior
        cv2.rectangle(frame_final, (10, 10), (450, 80), (0, 0, 0), -1) # Fondo negro HUD
        
        texto_rendimiento = f"FPS: {fps_actuales:.1f} | Latencia: {latencia:.1f} ms"
        texto_riesgo = f"ESTADO ADAS: {riesgo_global_frame}"
        
        cv2.putText(frame_final, texto_rendimiento, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame_final, texto_riesgo, (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_alerta_global, 2)

        # Guardar el frame con ambos análisis
        out.write(frame_final)

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"¡Video Híbrido Finalizado! Revisa: {ruta_video_salida}")

Cargando Cerebro 1 (Segmentación COCO)...
Cargando Cerebro 2 (Tu Modelo Personalizado para Señales)...
Iniciando procesamiento Multi-Modelo con Heurística de Riesgo...
¡Video Híbrido Finalizado! Revisa: ../reports/alertas_visuales/dia2segmentacion.mp4
